# Laboratorio: Diseñando Combos para "Café COVA" ☕

**Contexto:**
La gerencia de la cafetería de la facultad quiere aumentar sus ingresos. Saben que los estudiantes compran cosas en grupo, pero no saben exactamente qué. Tu trabajo como Ingeniero de Datos es leer los registros crudos de la caja registradora, transformar los datos a una matriz transaccional y aplicar el algoritmo Apriori para encontrar los "Combos" más rentables.

In [ ]:
%pip install mlxtend

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# FASE 1: EXTRACCIÓN (Datos de producción)
# El sistema punto de venta (POS) exporta los tickets como una lista de listas.
tickets_cafeteria = [
    ['Cafe Americano', 'Croissant', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Agua'],
    ['Cafe Americano', 'Croissant', 'Jugo de Naranja'],
    ['Te Verde', 'Galletas', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Galletas'],
    ['Cafe Americano', 'Croissant', 'Agua', 'Galletas'],
    ['Frappe Mocha', 'Brownie', 'Agua'],
    ['Te Verde', 'Ensalada', 'Agua'],
    ['Cafe Americano', 'Brownie', 'Jugo de Naranja'],
    ['Frappe Mocha', 'Muffin', 'Brownie']
]

### Reto 1: Ingeniería de Características (De Lista a Matriz)
El algoritmo Apriori no entiende listas de texto. Necesitas convertir `tickets_cafeteria` en un DataFrame de Pandas donde cada columna sea un producto y cada fila (ticket) tenga un `True` o `False` (One-Hot Encoding transaccional).

In [ ]:
# TU CÓDIGO AQUÍ:
# Utiliza TransactionEncoder para crear df_cafe_bool
te = TransactionEncoder()
te_ary = te.fit(tickets_cafeteria).transform(tickets_cafeteria)
df_cafe_bool = pd.DataFrame(te_ary, columns=te.columns_)
df_cafe_bool

### Reto 2: Extracción de Patrones Frecuentes
La gerencia te pide descartar las coincidencias raras. Extrae únicamente los `itemsets` (combinaciones de productos) que aparezcan en **al menos el 25% (0.25)** de los tickets.

In [ ]:
# TU CÓDIGO AQUÍ:
# Usa la función apriori() pasándole tu df_cafe_bool. Guarda el resultado en itemsets_cafe
itemsets_cafe = apriori(df_cafe_bool, min_support=0.25, use_colnames=True)
itemsets_cafe

### Reto 3: Creación de Reglas de Asociación
A partir de los itemsets frecuentes, genera las reglas de asociación. Filtra el resultado para mostrar **únicamente** aquellas reglas que tengan una `Confianza` superior al 60% (0.6).

In [ ]:
# TU CÓDIGO AQUÍ:
# Usa association_rules() y guarda el resultado en reglas_cafe
# Filtra las columnas para imprimir solo: antecedents, consequents, support, confidence, lift
reglas_cafe = association_rules(itemsets_cafe, metric="confidence", min_threshold=0.6)
reglas_cafe[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

### 🧠 Reto 4: Defensa de Ingeniería (Análisis de Negocio)

Revisa la tabla resultante de tus reglas y responde las siguientes preguntas para el gerente de la cafetería:

1. **La regla de mayor Confianza:** Identifica la regla que tiene el mayor nivel de Confianza. Traduce esa regla al español simple (ej. "Si un alumno compra X, existe un Y% de probabilidad de que también compre Z").
2. **El engaño del Soporte vs Lift:** ¿Qué producto de la cafetería se vende muchísimo (alto soporte), pero no deberíamos ponerlo en promoción (combo) con el Frappe Mocha porque su Lift es bajo o cercano a 1? Explica matemáticamente por qué.

1. **La regla de mayor Confianza:**
Las reglas con mayor confianza son `{Croissant} -> {Cafe Americano}` y `{Muffin} -> {Frappe Mocha}` con 100% de confianza. Es decir: si un alumno compra un Croissant, es 100% seguro que también va a pedir un Cafe Americano. Pasa exactamente igual con el Muffin: quien pide un muffin, siempre termina comprando un Frappe Mocha.

2. **El engaño del Soporte vs Lift:**
El producto es el **Agua**. Aunque tiene un soporte muy alto porque se vende en el 60% de los tickets, casi nunca coincide con el Frappe Mocha (solo 2 veces). Su Lift es de apenas 0.83 (menor a 1), lo que significa matemáticamente que comprar frappe no empuja a la gente a comprar agua; de hecho, coinciden menos de lo esperado. El agua ya se vende sola por necesidad, así que armar un combo con ella no crearía ventas nuevas y solo le restaría ganancia a la cafetería.